# Análise de dependencia da rede neural

## Coleta e Limpeza dos Dados

In [2]:
import numpy as np
import machinelearning_mnpkllps.Preprocessing as PP
import machinelearning_mnpkllps.Resampling as RA
import machinelearning_mnpkllps.NeuralNetwork as NN
import machinelearning_mnpkllps.Metrics as MT

ModuleNotFoundError: No module named 'machinelearning_mnpkllps.metricas'

In [ ]:
# Leitura dos dados
dados = np.genfromtxt('../data/RESULTADOS_2025.csv', 
                      delimiter=';', 
                      skip_header=1, 
                      encoding='latin-1', 
                      usecols=(12, 22, 23, 24, 25, 30, 36, 37, 38, 39, 40))

# Colunas selecionadas:
# UF, NOTA_CN, NOTA_CH, NOTA_LC, NOTA_MT, LINGUA, NOTA_R1, NOTA_R2, NOTA_R3, NOTA_R4, NOTA_R5

In [ ]:
# Leitura dos dados
R_total = np.genfromtxt('../data/RESULTADOS_2025.csv', 
                      delimiter=';', 
                      skip_header=1, 
                      encoding='latin-1', 
                      usecols=(41))

# Colunas selecionadas:
# NOTA_RT

In [ ]:
data = dados.copy()
rng = np.random.default_rng(450)

linhas_com_nan = np.any(np.isnan(data), axis=1)

linhas_validas = ~linhas_com_nan

n = int(3e5)
clean_data = data[linhas_validas]

_, clean_data, _, _ = PP.train_test_split(clean_data, clean_data, test_size=0.10)




print(f"Número de linhas originais: {data.shape[0]}")
print(f"Número de linhas da base limpa: {clean_data.shape[0]}")

Número de linhas originais: 4810772
Número de linhas da base limpa: 324435


In [ ]:
clean_data = PP.ColumnCategoricalEncoder([0], drop_first=True).fit_transform(clean_data)

In [ ]:
col = {
    'UF': 0,
    'NOTA_CN': 26, 
    'NOTA_CH': 27, 
    'NOTA_LC': 28, 
    'NOTA_MT': 29, 
    'LINGUA': 30, 
    'NOTA_R1': 31, 
    'NOTA_R2': 32, 
    'NOTA_R3': 33, 
    'NOTA_R4': 34, 
    'NOTA_R5': 35
}

In [ ]:
X = clean_data[:, :col['LINGUA']+1]
y = clean_data[:, col['NOTA_R1']:]

display(X)
display(y)

array([[  0. ,   0. ,   0. , ..., 432.9, 340. ,   0. ],
       [  0. ,   0. ,   0. , ..., 613.7, 539. ,   0. ],
       [  0. ,   0. ,   0. , ..., 377.8, 453.2,   0. ],
       ...,
       [  0. ,   0. ,   0. , ..., 505.3, 454.5,   0. ],
       [  0. ,   0. ,   0. , ..., 420. , 449.9,   1. ],
       [  0. ,   0. ,   0. , ..., 620.2, 741.1,   0. ]],
      shape=(300000, 31))

array([[  0.,   0.,   0.,   0.,   0.],
       [140., 120., 120., 120.,  80.],
       [  0.,   0.,   0.,   0.,   0.],
       ...,
       [100., 140., 140., 120., 140.],
       [100., 160., 120., 120.,  20.],
       [160., 200., 180., 180., 200.]], shape=(300000, 5))

In [ ]:
x_train, x_aux, y_train, y_aux = PP.train_test_split(X, y, test_size=0.40)
x_valid, x_test, y_valid, y_test = PP.train_test_split(x_aux, y_aux, test_size=0.50)

display(x_train.shape, x_test.shape)
display(x_valid.shape, x_valid.shape)
display(y_test.shape, y_test.shape)

(180000, 31)

(60000, 31)

(60000, 31)

(60000, 31)

(60000, 5)

(60000, 5)

In [ ]:
x_train = PP.MinMaxScaler().fit_transform(x_train)
x_valid = PP.MinMaxScaler().fit_transform(x_valid)
x_test = PP.MinMaxScaler().fit_transform(x_test)

y_train = PP.MinMaxScaler().fit_transform(y_train)
y_valid = PP.MinMaxScaler().fit_transform(y_valid)
y_test = PP.MinMaxScaler().fit_transform(y_test)

## Realizando a regração separadamente para cada variavel

In [ ]:
NNs = []
for i in range(5):
    nn = NN.NeuralNetwork([31, 64, 32, 16, 1], 'ReLU', 'sigmoid', bias_interval=(0, 0), weight_interval=(-1, 1), seed=782+i)
    nn.fit(x_train, y_train[:, i].reshape(-1, 1), learn_rate=0.001, batch=32, epochs=200, debug = True, tol=1e-8)

    NNs.append(nn)

Iniciando epoca 0, com erro 0.3635190704440238
Iniciando epoca 1, com erro 0.35005444334505376
Iniciando epoca 2, com erro 0.14934095019611315
Iniciando epoca 3, com erro 0.08978729291304852
Iniciando epoca 4, com erro 0.0790898944133926
Iniciando epoca 5, com erro 0.05481736021814844
Iniciando epoca 6, com erro 0.04457813084030421
Iniciando epoca 7, com erro 0.03690331604541697
Iniciando epoca 8, com erro 0.03383118665042425
Iniciando epoca 9, com erro 0.03221009753103899
Iniciando epoca 10, com erro 0.031314467186716155
Iniciando epoca 11, com erro 0.030563425931287514
Iniciando epoca 12, com erro 0.030081244601003376
Iniciando epoca 13, com erro 0.029694250712604207
Iniciando epoca 14, com erro 0.029464917915240422
Iniciando epoca 15, com erro 0.02910306543419742
Iniciando epoca 16, com erro 0.028880732881881248
Iniciando epoca 17, com erro 0.028693897452280608
Iniciando epoca 18, com erro 0.02851084265976594
Iniciando epoca 19, com erro 0.028364200264212963
Iniciando epoca 20, com 

In [ ]:
R1_train = y_train[:, 0].reshape(-1, 1)
R2_train = y_train[:, 1].reshape(-1, 1)
R3_train = y_train[:, 2].reshape(-1, 1)
R4_train = y_train[:, 3].reshape(-1, 1)
R5_train = y_train[:, 4].reshape(-1, 1)
RT_train = R1_train + R2_train + R3_train + R4_train + R5_train

R1_prev_train = NNs[0].prevision(x_train)
R2_prev_train = NNs[1].prevision(x_train)
R3_prev_train = NNs[2].prevision(x_train)
R4_prev_train = NNs[3].prevision(x_train)
R5_prev_train = NNs[4].prevision(x_train)
RT_prev_train = R1_prev_train + R2_prev_train + R3_prev_train + R4_prev_train + R5_prev_train

print(f'Erro treino Nota R1: {MT.RMSE(R1_prev_train, R1_train)*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R2_prev_train, R2_train)*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R3_prev_train, R3_train)*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R4_prev_train, R4_train)*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R5_prev_train, R5_train)*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_train, RT_train)*200}')

Erro treino Nota R1: 32.087920770602814
Erro treino Nota R2: 37.69308956938941
Erro treino Nota R3: 33.26997849573482
Erro treino Nota R4: 37.44177000222383
Erro treino Nota R5: 52.85467227114864
Erro treino Nota RT: 167.43279571527498


In [ ]:
R1_valid = y_valid[:, 0].reshape(-1, 1)
R2_valid = y_valid[:, 1].reshape(-1, 1)
R3_valid = y_valid[:, 2].reshape(-1, 1)
R4_valid = y_valid[:, 3].reshape(-1, 1)
R5_valid = y_valid[:, 4].reshape(-1, 1)
RT_valid = R1_valid + R2_valid + R3_valid + R4_valid + R5_valid

R1_prev_valid = NNs[0].prevision(x_valid)
R2_prev_valid = NNs[1].prevision(x_valid)
R3_prev_valid = NNs[2].prevision(x_valid)
R4_prev_valid = NNs[3].prevision(x_valid)
R5_prev_valid = NNs[4].prevision(x_valid)
RT_prev_valid = R1_prev_valid + R2_prev_valid + R3_prev_valid + R4_prev_valid + R5_prev_valid

print(f'Erro treino Nota R1: {MT.RMSE(R1_prev_valid, R1_valid)*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R2_prev_valid, R2_valid)*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R3_prev_valid, R3_valid)*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R4_prev_valid, R4_valid)*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R5_prev_valid, R5_valid)*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_valid, RT_valid)*200}')

Erro treino Nota R1: 31.895860720215268
Erro treino Nota R2: 37.646734067746735
Erro treino Nota R3: 33.16152725736022
Erro treino Nota R4: 37.313295186502174
Erro treino Nota R5: 52.70250346934774
Erro treino Nota RT: 166.72380065250596


In [ ]:
R1_test = y_test[:, 0].reshape(-1, 1)
R2_test = y_test[:, 1].reshape(-1, 1)
R3_test = y_test[:, 2].reshape(-1, 1)
R4_test = y_test[:, 3].reshape(-1, 1)
R5_test = y_test[:, 4].reshape(-1, 1)
RT_test = R1_test + R2_test + R3_test + R4_test + R5_test

R1_prev_test = NNs[0].prevision(x_test)
R2_prev_test = NNs[1].prevision(x_test)
R3_prev_test = NNs[2].prevision(x_test)
R4_prev_test = NNs[3].prevision(x_test)
R5_prev_test = NNs[4].prevision(x_test)
RT_prev_test = R1_prev_test + R2_prev_test + R3_prev_test + R4_prev_test + R5_prev_test

print(f'Erro treino Nota R1: {MT.RMSE(R1_prev_test, R1_test)*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R2_prev_test, R2_test)*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R3_prev_test, R3_test)*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R4_prev_test, R4_test)*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R5_prev_test, R5_test)*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_test, RT_test)*200}')


Erro treino Nota R1: 31.996115347191644
Erro treino Nota R2: 37.67216346636487
Erro treino Nota R3: 33.40440393586836
Erro treino Nota R4: 37.48329207715167
Erro treino Nota R5: 53.07332274920896
Erro treino Nota RT: 167.6380845689532


In [ ]:
print("Nota real:")
print(RT_train.min(), RT_train.max(), RT_train.mean())

print("Nota prevista:")
print(RT_prev_train.min(), RT_prev_train.max(), RT_prev_train.mean())

Nota real:
0.0 4.9 2.9486600000000003
Nota prevista:
2.945979558328916 2.9531509891676593 2.9488861022757895


## Realizando a regração para todas juntas

In [ ]:
nn = NN.NeuralNetwork([31, 64, 32, 5], 'ReLU', 'sigmoid', bias_interval=(0, 0), weight_interval=(-1, 1), seed=147)
nn.fit(x_train, y_train, learn_rate=0.0005, batch=64, epochs=500, debug = True, tol=1e-8)

Iniciando epoca 0, com erro 0.192043844918282
Iniciando epoca 1, com erro 0.14934946690962217
Iniciando epoca 2, com erro 0.13375737306637722
Iniciando epoca 3, com erro 0.12428235431706262
Iniciando epoca 4, com erro 0.11655336451225924
Iniciando epoca 5, com erro 0.10977138578441514
Iniciando epoca 6, com erro 0.10383108415667262
Iniciando epoca 7, com erro 0.09865965331633623
Iniciando epoca 8, com erro 0.09411568976844853
Iniciando epoca 9, com erro 0.09017587713231795
Iniciando epoca 10, com erro 0.08657068966919831
Iniciando epoca 11, com erro 0.08264016341903899
Iniciando epoca 12, com erro 0.07896475234799169
Iniciando epoca 13, com erro 0.07654190593338554
Iniciando epoca 14, com erro 0.07450942136319932
Iniciando epoca 15, com erro 0.07260593228952739
Iniciando epoca 16, com erro 0.07074690168649343
Iniciando epoca 17, com erro 0.0689123866082979
Iniciando epoca 18, com erro 0.06708347967349729
Iniciando epoca 19, com erro 0.06523047225794068
Iniciando epoca 20, com erro 0.06

In [ ]:
R_train = y_train
RT_train = np.sum(R_train, axis=1)

R_prev_train = nn.prevision(x_train)
RT_prev_train = np.sum(R_prev_train, axis=1)

print(f'Erro treino Nota R1: {MT.RMSE(R_prev_train[:, 0], R_train[:, 0])*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R_prev_train[:, 1], R_train[:, 1])*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R_prev_train[:, 2], R_train[:, 2])*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R_prev_train[:, 3], R_train[:, 3])*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R_prev_train[:, 4], R_train[:, 4])*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_train, RT_train)*200}')

Erro treino Nota R1: 31.974850376429753
Erro treino Nota R2: 37.58760391395205
Erro treino Nota R3: 33.072464600286494
Erro treino Nota R4: 37.379635182763835
Erro treino Nota R5: 52.73932580650794
Erro treino Nota RT: 167.4502847511998


In [ ]:
R_valid = y_valid
RT_valid = np.sum(R_valid, axis=1)

R_prev_valid = nn.prevision(x_valid)
RT_prev_valid = np.sum(R_prev_valid, axis=1)

print(f'Erro treino Nota R1: {MT.RMSE(R_prev_valid[:, 0], R_valid[:, 0])*200}')
print(f'Erro treino Nota R2: {MT.RMSE(R_prev_valid[:, 1], R_valid[:, 1])*200}')
print(f'Erro treino Nota R3: {MT.RMSE(R_prev_valid[:, 2], R_valid[:, 2])*200}')
print(f'Erro treino Nota R4: {MT.RMSE(R_prev_valid[:, 3], R_valid[:, 3])*200}')
print(f'Erro treino Nota R5: {MT.RMSE(R_prev_valid[:, 4], R_valid[:, 4])*200}')
print(f'Erro treino Nota RT: {MT.RMSE(RT_prev_valid, RT_valid)*200}')

Erro treino Nota R1: 31.77959974191561
Erro treino Nota R2: 37.50440764189777
Erro treino Nota R3: 32.98012798414198
Erro treino Nota R4: 37.2302836678093
Erro treino Nota R5: 52.560867275390756
Erro treino Nota RT: 166.7131203910284


In [ ]:
for i in range(5):
    baseline = np.full_like(y_valid[:, i], np.mean(y_train[:, i]))

    print(
        f"R{i+1}:",
        MT.RMSE(baseline, y_valid[:, i])
    )

R1: 0.18673709854125115
R2: 0.21612342461349873
R3: 0.19504290208253613
R4: 0.22307189134591113
R5: 0.31859263403025084
